In [1]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
SERPAPI_KEY = os.getenv("SERPAPI_KEY")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import folium
import math
import json
import requests
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
import warnings
warnings.filterwarnings('ignore')

client = OpenAI()
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
print("All imports successful")
print(f"SerpAPI key: {'loaded' if SERPAPI_KEY else 'MISSING'}")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful
SerpAPI key: loaded


In [2]:
# Cell 2 — Import all IFR math from Project 26
# (reuse the entire math engine)

def haversine(lat1, lon1, lat2, lon2):
    R = 3440.065
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def initial_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def e6b_wind_correction(tas_kts, wind_dir, wind_spd, true_course):
    tc_rad = math.radians(true_course)
    wd_rad = math.radians(wind_dir)
    hwc    = wind_spd * math.cos(wd_rad - tc_rad)
    xwc    = wind_spd * math.sin(wd_rad - tc_rad)
    wca    = math.degrees(math.asin(xwc / tas_kts)) if abs(xwc/tas_kts) <= 1 else 0
    gs     = tas_kts * math.cos(math.radians(wca)) - hwc
    mh     = (true_course + wca - 14.0 + 360) % 360
    return round(wca, 1), round(gs, 1), round(mh, 1)

def density_altitude(elevation_ft, temp_c, altimeter_inhg=29.92):
    pressure_alt = elevation_ft + (29.92 - altimeter_inhg) * 1000
    isa_temp_c   = 15 - (0.00198 * elevation_ft)
    return round(pressure_alt + 118.8 * (temp_c - isa_temp_c), 0)

def holding_pattern_entry(inbound_course, aircraft_heading):
    rel = (aircraft_heading - inbound_course + 360) % 360
    if 0 <= rel <= 70 or 290 <= rel <= 360:
        return "Direct", f"Fly direct to fix, turn RIGHT to outbound {(inbound_course+180)%360}°"
    elif 71 <= rel <= 180:
        return "Teardrop", f"Cross fix, fly {(inbound_course+150)%360}° for 1 min, turn RIGHT to {inbound_course}°"
    else:
        return "Parallel", f"Cross fix, turn LEFT to {(inbound_course+180)%360}° for 1 min, turn RIGHT to {inbound_course}°"

airports = {
    'KBOS': {'name': 'Boston Logan',       'lat': 42.3656, 'lon': -71.0096, 'elev': 19,   'class': 'B'},
    'KJFK': {'name': 'JFK International',  'lat': 40.6413, 'lon': -73.7781, 'elev': 13,   'class': 'B'},
    'KORH': {'name': 'Worcester Regional', 'lat': 42.2673, 'lon': -71.8757, 'elev': 1009, 'class': 'D'},
    'KBDL': {'name': 'Bradley Intl',       'lat': 41.9389, 'lon': -72.6832, 'elev': 173,  'class': 'C'},
    'KMHT': {'name': 'Manchester-Boston',  'lat': 42.9326, 'lon': -71.4357, 'elev': 235,  'class': 'C'},
    'KPVD': {'name': 'Providence TF Green','lat': 41.7272, 'lon': -71.4284, 'elev': 55,   'class': 'C'},
    'KLGA': {'name': 'LaGuardia',          'lat': 40.7772, 'lon': -73.8726, 'elev': 21,   'class': 'B'},
    'KORD': {'name': 'Chicago O\'Hare',    'lat': 41.9742, 'lon': -87.9073, 'elev': 672,  'class': 'B'},
    'KATL': {'name': 'Atlanta Hartsfield', 'lat': 33.6407, 'lon': -84.4277, 'elev': 1026, 'class': 'B'},
    'KMIA': {'name': 'Miami Intl',         'lat': 25.7959, 'lon': -80.2870, 'elev': 8,    'class': 'B'},
    'KBFI': {'name': 'Boeing Field',       'lat': 47.5300, 'lon': -122.3019,'elev': 21,   'class': 'D'},
    'KSEA': {'name': 'Seattle-Tacoma',     'lat': 47.4502, 'lon': -122.3088,'elev': 433,  'class': 'B'},
}

print("IFR math engine loaded")
print(f"Airports in database: {len(airports)}")

IFR math engine loaded
Airports in database: 12


In [3]:
# Cell 3 — VFR Airspace Classification & Requirements
def classify_airspace(airport_icao):
    """Return VFR requirements for airport airspace class"""
    airspace_rules = {
        'B': {
            'name': 'Class B',
            'clearance': 'ATC clearance required',
            'visibility_sm': 3,
            'cloud_clearance': 'Clear of clouds',
            'altitude_limit': '10,000 ft MSL',
            'transponder': 'Mode C required',
            'notes': 'Most complex airspace — major airports'
        },
        'C': {
            'name': 'Class C',
            'clearance': 'Two-way radio contact required',
            'visibility_sm': 3,
            'cloud_clearance': '500 below, 1000 above, 2000 horizontal',
            'altitude_limit': '4,000 ft AGL',
            'transponder': 'Mode C required',
            'notes': 'Medium hub airports — 5/10 NM rings'
        },
        'D': {
            'name': 'Class D',
            'clearance': 'Two-way radio contact required',
            'visibility_sm': 3,
            'cloud_clearance': '500 below, 1000 above, 2000 horizontal',
            'altitude_limit': '2,500 ft AGL',
            'transponder': 'Not required (recommended)',
            'notes': 'Smaller airports with control tower'
        },
        'E': {
            'name': 'Class E',
            'clearance': 'None required',
            'visibility_sm': 3,
            'cloud_clearance': '500 below, 1000 above, 2000 horizontal',
            'altitude_limit': 'Varies',
            'transponder': 'Above 10,000 MSL only',
            'notes': 'Controlled airspace without tower'
        },
        'G': {
            'name': 'Class G',
            'clearance': 'None required',
            'visibility_sm': 1,
            'cloud_clearance': 'Clear of clouds (below 1200 AGL day)',
            'altitude_limit': 'Up to Class E floor',
            'transponder': 'Not required',
            'notes': 'Uncontrolled airspace'
        }
    }
    
    if airport_icao in airports:
        cls = airports[airport_icao]['class']
        rules = airspace_rules.get(cls, airspace_rules['E'])
        return {
            'airport': airport_icao,
            'name': airports[airport_icao]['name'],
            'airspace_class': cls,
            **rules
        }
    return {'error': f'Airport {airport_icao} not found'}

def vfr_weather_check(visibility_sm, ceiling_ft, altitude_ft, airspace_class):
    """
    Check if VFR flight is legal given weather conditions
    Returns go/no-go recommendation with explanation
    """
    results = []
    
    # Basic VFR minimums
    if visibility_sm < 3:
        results.append(f"❌ Visibility {visibility_sm}SM below 3SM VFR minimum")
    else:
        results.append(f"✓ Visibility {visibility_sm}SM meets VFR minimum")
    
    # Cloud clearance check
    if ceiling_ft < 1000:
        results.append(f"❌ Ceiling {ceiling_ft}ft — below 1000ft AGL minimum")
    elif ceiling_ft < 3000:
        results.append(f"⚠ Ceiling {ceiling_ft}ft — marginal VFR conditions")
    else:
        results.append(f"✓ Ceiling {ceiling_ft}ft — good VFR conditions")
    
    # Class B special check
    if airspace_class == 'B' and visibility_sm < 3:
        results.append("❌ Class B requires 3SM visibility minimum")
    
    # Night VFR
    if ceiling_ft >= 1000 and visibility_sm >= 3:
        verdict = "GO — VFR conditions met"
    elif ceiling_ft >= 500 and visibility_sm >= 1:
        verdict = "MARGINAL VFR — proceed with caution"
    else:
        verdict = "NO-GO — below VFR minimums"
    
    return {'verdict': verdict, 'checks': results}

# Test airspace classification
print("=== VFR AIRSPACE CLASSIFICATION ===\n")
for icao in ['KBOS', 'KBDL', 'KORH']:
    info = classify_airspace(icao)
    print(f"{icao} — {info['name']}")
    print(f"  Class:        {info['airspace_class']} ({info['name']})")
    print(f"  Clearance:    {info['clearance']}")
    print(f"  Visibility:   {info['visibility_sm']} SM")
    print(f"  Clouds:       {info['cloud_clearance']}")
    print(f"  Transponder:  {info['transponder']}")
    print()

# Test VFR weather check
print("=== VFR WEATHER CHECK ===\n")
scenarios = [
    (5, 3000, 3500, 'B', "Good day"),
    (2, 800,  1500, 'C', "Marginal conditions"),
    (1, 400,  800,  'D', "Below minimums"),
]
for vis, ceil, alt, cls, label in scenarios:
    result = vfr_weather_check(vis, ceil, alt, cls)
    print(f"{label}: {result['verdict']}")
    for check in result['checks']:
        print(f"  {check}")
    print()

=== VFR AIRSPACE CLASSIFICATION ===

KBOS — Class B
  Class:        B (Class B)
  Clearance:    ATC clearance required
  Visibility:   3 SM
  Clouds:       Clear of clouds
  Transponder:  Mode C required

KBDL — Class C
  Class:        C (Class C)
  Clearance:    Two-way radio contact required
  Visibility:   3 SM
  Clouds:       500 below, 1000 above, 2000 horizontal
  Transponder:  Mode C required

KORH — Class D
  Class:        D (Class D)
  Clearance:    Two-way radio contact required
  Visibility:   3 SM
  Clouds:       500 below, 1000 above, 2000 horizontal
  Transponder:  Not required (recommended)

=== VFR WEATHER CHECK ===

Good day: GO — VFR conditions met
  ✓ Visibility 5SM meets VFR minimum
  ✓ Ceiling 3000ft — good VFR conditions

Marginal conditions: MARGINAL VFR — proceed with caution
  ❌ Visibility 2SM below 3SM VFR minimum
  ❌ Ceiling 800ft — below 1000ft AGL minimum

Below minimums: NO-GO — below VFR minimums
  ❌ Visibility 1SM below 3SM VFR minimum
  ❌ Ceiling 400ft 

In [4]:
# Cell 4 — Interactive VFR Sectional Chart with Folium
def create_vfr_sectional_map(center_airport='KBOS', zoom=9):
    """
    Create interactive VFR sectional chart using FAA tile server
    Overlays airport markers and airspace boundaries
    """
    center = airports[center_airport]
    m = folium.Map(
        location=[center['lat'], center['lon']],
        zoom_start=zoom,
        tiles=None
    )
    
    # FAA VFR Sectional Chart tiles (free, public)
    folium.TileLayer(
        tiles='https://vfrmap.com/20250101/tiles/vfrc/{z}/{y}/{x}.jpg',
        attr='FAA VFR Sectional Chart',
        name='VFR Sectional',
        overlay=False,
        control=True
    ).add_to(m)
    
    # Fallback: OpenStreetMap
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='Street Map',
        overlay=False,
        control=True
    ).add_to(m)
    
    # Add airport markers
    airspace_colors = {
        'B': 'blue', 'C': 'purple',
        'D': 'darkblue', 'E': 'green', 'G': 'gray'
    }
    
    for icao, info in airports.items():
        color = airspace_colors.get(info['class'], 'red')
        folium.Marker(
            location=[info['lat'], info['lon']],
            popup=folium.Popup(
                f"<b>{icao}</b><br>{info['name']}<br>"
                f"Class {info['class']}<br>Elev: {info['elev']}ft",
                max_width=200
            ),
            tooltip=f"{icao} — Class {info['class']}",
            icon=folium.Icon(color=color, icon='plane', prefix='fa')
        ).add_to(m)
    
    # Add route line KBOS → KJFK
    route_points = [
        [airports['KBOS']['lat'], airports['KBOS']['lon']],
        [airports['KLGA']['lat'], airports['KLGA']['lon']],
        [airports['KJFK']['lat'], airports['KJFK']['lon']],
    ]
    folium.PolyLine(
        route_points,
        color='red',
        weight=3,
        opacity=0.8,
        tooltip='KBOS → KLGA → KJFK'
    ).add_to(m)
    
    folium.LayerControl().add_to(m)
    
    # Save map
    output_dir = r'C:\Users\Gurveer\ds-portfolio\project-28-pilotgpt-v3\outputs'
    os.makedirs(output_dir, exist_ok=True)
    map_path = f'{output_dir}\\vfr_sectional_map.html'
    m.save(map_path)
    print(f"VFR sectional map saved: {map_path}")
    print("Open the HTML file in your browser to view the interactive map")
    return map_path

import os
map_path = create_vfr_sectional_map('KBOS', zoom=8)

VFR sectional map saved: C:\Users\Gurveer\ds-portfolio\project-28-pilotgpt-v3\outputs\vfr_sectional_map.html
Open the HTML file in your browser to view the interactive map


In [5]:
# Cell 5 — SerpAPI Live Weather & NOTAM Lookup
def get_aviation_weather(airport_icao):
    """Fetch live aviation weather via SerpAPI"""
    if not SERPAPI_KEY:
        return {"error": "No SerpAPI key found in .env"}
    
    try:
        response = requests.get(
            "https://serpapi.com/search",
            params={
                "api_key": SERPAPI_KEY,
                "engine":  "google",
                "q":       f"{airport_icao} METAR weather aviation current",
                "num":     3
            },
            timeout=15
        )
        data     = response.json()
        snippets = []
        
        if "organic_results" in data:
            for r in data["organic_results"][:3]:
                snippets.append({
                    "title":   r.get("title", ""),
                    "snippet": r.get("snippet", ""),
                    "link":    r.get("link", "")
                })
        
        return {
            "airport":  airport_icao,
            "results":  snippets,
            "status":   "live data fetched"
        }
    except Exception as e:
        return {"error": str(e)}

def get_notams(airport_icao):
    """Fetch live NOTAMs via SerpAPI"""
    if not SERPAPI_KEY:
        return {"error": "No SerpAPI key"}
    
    try:
        response = requests.get(
            "https://serpapi.com/search",
            params={
                "api_key": SERPAPI_KEY,
                "engine":  "google",
                "q":       f"NOTAM {airport_icao} airport notice pilots current 2025",
                "num":     3
            },
            timeout=15
        )
        data     = response.json()
        snippets = []
        
        if "organic_results" in data:
            for r in data["organic_results"][:3]:
                snippets.append({
                    "title":   r.get("title", ""),
                    "snippet": r.get("snippet", "")
                })
        
        return {"airport": airport_icao, "notams": snippets}
    except Exception as e:
        return {"error": str(e)}

# Test live lookups
print("=== LIVE WEATHER LOOKUP: KBOS ===")
wx = get_aviation_weather("KBOS")
if "results" in wx:
    for r in wx["results"]:
        print(f"\n  {r['title']}")
        print(f"  {r['snippet'][:120]}...")
else:
    print(f"  Error: {wx.get('error')}")

print("\n=== LIVE NOTAM LOOKUP: KBOS ===")
notams = get_notams("KBOS")
if "notams" in notams:
    for n in notams["notams"]:
        print(f"\n  {n['title']}")
        print(f"  {n['snippet'][:120]}...")
else:
    print(f"  Error: {notams.get('error')}")

=== LIVE WEATHER LOOKUP: KBOS ===

  METAR and TAF Data
  Clouds: few clouds at 2,000 ft, scattered clouds at 25,000 ft ; TAF for: Boston/Logan Intl ; Text: TAF KBOS 281720Z 2818...

  KBOS (BOS) Boston Logan International Airport, METAR ...
  Wind 300° 11kt. Visibility 10sm. Clouds few 20000ft. Temperature 14°C, dew point -3°C. Altimeter 29.74inHg. Remark: AO2 ...

  FAA WeatherCams
  General Edward Lawrence Logan Intl, MA (KBOS) ; Winds: From ENE (070°) at 10 kts ; Visibility: 2 miles ; Ceiling: 600 ft...

=== LIVE NOTAM LOOKUP: KBOS ===

  🫡🫡🫡NOTAMs (Notices to Airmen) issued worldwide, ...
  NOTAMs (Notices to Airmen) issued worldwide, highlighting potential hazards, airspace restrictions, facility changes, or...

  Graphic TFRs - Federal Aviation Administration
  Operating Restrictions and Requirements ; No pilots may operate an aircraft in the areas covered by this NOTAM (except a...

  flight information publication terminal procedures publication
  These specifications have bee

In [6]:
# Cell 6 — Build PilotGPT v3 Agent with 7 Tools
@tool
def get_airspace_info(airport_icao: str) -> str:
    """Get VFR airspace classification and requirements for an airport."""
    info = classify_airspace(airport_icao.upper())
    return json.dumps(info)

@tool
def check_vfr_conditions(visibility_sm: float, ceiling_ft: int,
                          altitude_ft: int, airspace_class: str) -> str:
    """Check if VFR flight is legal given weather conditions and airspace class."""
    result = vfr_weather_check(visibility_sm, ceiling_ft, altitude_ft, airspace_class)
    return json.dumps(result)

@tool
def calculate_holding_entry(inbound_course: int, aircraft_heading: int) -> str:
    """Calculate IFR holding pattern entry type."""
    entry, desc = holding_pattern_entry(inbound_course, aircraft_heading)
    return json.dumps({"entry_type": entry, "procedure": desc})

@tool
def calculate_wind_correction(tas_kts: float, wind_dir: int,
                               wind_spd: int, true_course: int) -> str:
    """Calculate wind correction angle, ground speed, and magnetic heading."""
    wca, gs, mh = e6b_wind_correction(tas_kts, wind_dir, wind_spd, true_course)
    return json.dumps({"wca_deg": wca, "ground_speed_kts": gs, "mag_heading_deg": mh})

@tool
def calculate_density_altitude_tool(elevation_ft: int, temp_c: float,
                                     altimeter_inhg: float = 29.92) -> str:
    """Calculate density altitude for performance planning."""
    da = density_altitude(elevation_ft, temp_c, altimeter_inhg)
    return json.dumps({"density_altitude_ft": da,
                       "pressure_altitude_ft": elevation_ft + (29.92 - altimeter_inhg) * 1000})

@tool
def get_live_weather(airport_icao: str) -> str:
    """Fetch live aviation weather for an airport using web search."""
    result = get_aviation_weather(airport_icao.upper())
    return json.dumps(result)

@tool
def get_airport_details(airport_icao: str) -> str:
    """Get airport information including airspace class and elevation."""
    icao = airport_icao.upper()
    if icao in airports:
        info = airports[icao].copy()
        dist_from_bos = haversine(
            airports['KBOS']['lat'], airports['KBOS']['lon'],
            info['lat'], info['lon']
        )
        info['distance_from_KBOS_nm'] = round(dist_from_bos, 1)
        return json.dumps({"icao": icao, **info})
    return json.dumps({"error": f"Airport {icao} not in database"})

tools = [
    get_airspace_info, check_vfr_conditions,
    calculate_holding_entry, calculate_wind_correction,
    calculate_density_altitude_tool, get_live_weather,
    get_airport_details
]

pilotgpt_v3 = create_react_agent(llm, tools=tools)
print(f"PilotGPT v3 ready — {len(tools)} tools loaded")
print(f"Tools: {[t.name for t in tools]}")

PilotGPT v3 ready — 7 tools loaded
Tools: ['get_airspace_info', 'check_vfr_conditions', 'calculate_holding_entry', 'calculate_wind_correction', 'calculate_density_altitude_tool', 'get_live_weather', 'get_airport_details']


In [7]:
# Cell 7 — PilotGPT v3 Live Queries
queries = [
    "I'm planning a VFR flight into KBDL. What airspace class is it and what are the VFR requirements?",
    "Current conditions at KBDL: visibility 4SM, ceiling 2500ft, I'm at 3000ft. Can I fly VFR?",
    "I'm flying VFR from KBOS to KJFK. TAS 110 knots, winds 250 at 20 knots. What's my wind correction and ground speed?",
    "What is the density altitude at KBOS today with OAT 28°C and altimeter 29.85?",
    "I'm entering a holding pattern. Inbound course is 090, my current heading is 200 degrees. What entry should I use?",
    "Give me a complete VFR preflight briefing for a flight from KBOS to KORH. Include airspace, distance, and any key considerations."
]

print("="*60)
print("  PILOTGPT v3 — VFR & IFR FLIGHT ASSISTANT")
print("="*60)

for q in queries:
    print(f"\nPilot: {q}")
    print("-"*60)
    result = pilotgpt_v3.invoke({
        "messages": [{"role": "user", "content": q}]
    })
    print(f"PilotGPT: {result['messages'][-1].content}")
    print()

  PILOTGPT v3 — VFR & IFR FLIGHT ASSISTANT

Pilot: I'm planning a VFR flight into KBDL. What airspace class is it and what are the VFR requirements?
------------------------------------------------------------
PilotGPT: KBDL (Bradley International Airport) is classified as Class C airspace. Here are the VFR requirements for flying in this airspace:

- **Clearance**: Two-way radio contact is required.
- **Visibility**: Minimum visibility of 3 statute miles.
- **Cloud Clearance**: You must maintain:
  - 500 feet below clouds
  - 1,000 feet above clouds
  - 2,000 feet horizontal distance from clouds
- **Altitude Limit**: Up to 4,000 feet AGL (Above Ground Level).
- **Transponder**: Mode C transponder is required.

Additionally, KBDL is a medium hub airport with a 5/10 NM ring structure.


Pilot: Current conditions at KBDL: visibility 4SM, ceiling 2500ft, I'm at 3000ft. Can I fly VFR?
------------------------------------------------------------
PilotGPT: You can fly VFR at KBDL under the c

In [8]:
# Cell 8 — Export & Summary
import os

output_dir = r'C:\Users\Gurveer\ds-portfolio\project-28-pilotgpt-v3\outputs'
os.makedirs(output_dir, exist_ok=True)

# Save airport database
pd.DataFrame(airports).T.to_csv(f'{output_dir}\\airport_database.csv')

# Save airspace rules
airspace_results = []
for icao in airports.keys():
    info = classify_airspace(icao)
    airspace_results.append(info)
pd.DataFrame(airspace_results).to_csv(
    f'{output_dir}\\airspace_classification.csv', index=False
)

# VFR weather scenarios
scenarios_df = pd.DataFrame([
    {'visibility_sm': 5, 'ceiling_ft': 3000, 'class': 'B', 'verdict': 'GO'},
    {'visibility_sm': 2, 'ceiling_ft': 800,  'class': 'C', 'verdict': 'MARGINAL'},
    {'visibility_sm': 1, 'ceiling_ft': 400,  'class': 'D', 'verdict': 'NO-GO'},
])
scenarios_df.to_csv(f'{output_dir}\\vfr_scenarios.csv', index=False)

print("=== PilotGPT v3 Summary ===")
print(f"  Tools:            {len(tools)}")
print(f"  Airports in DB:   {len(airports)}")
print(f"  Airspace classes: A, B, C, D, E, G")
print(f"  IFR features:     Holding patterns, wind correction, density altitude")
print(f"  VFR features:     Airspace classification, weather go/no-go")
print(f"  Live data:        SerpAPI weather and NOTAM lookup")
print(f"  Map:              Interactive FAA VFR sectional chart")
print(f"\nExports saved to outputs/")

=== PilotGPT v3 Summary ===
  Tools:            7
  Airports in DB:   12
  Airspace classes: A, B, C, D, E, G
  IFR features:     Holding patterns, wind correction, density altitude
  VFR features:     Airspace classification, weather go/no-go
  Live data:        SerpAPI weather and NOTAM lookup
  Map:              Interactive FAA VFR sectional chart

Exports saved to outputs/
